In [ ]:
import pandas as pd
import numpy as np
import gzip
import json

In [ ]:
data_path = "amazon_mt_2023/Movies_and_TV.jsonl.gz"
meta_path = "amazon_mt_2023/meta_Movies_and_TV.jsonl.gz"

In [ ]:
def parse(path):
  g = gzip.open(path, 'rb')
  for l in g:
    yield json.loads(l)

def getDF(path):
  i = 0
  df = {}
  for d in parse(path):
    df[i] = d
    i += 1
  return pd.DataFrame.from_dict(df, orient='index')

In [ ]:
df_main = getDF(data_path)
df_meta = getDF(meta_path)

In [ ]:
len(df_main), len(df_meta)

In [ ]:
df_meta.head(5)

In [ ]:
df_main.head(5)

In [ ]:
title_parent_asin_dict = df_meta.set_index('parent_asin')['title'].to_dict()


In [ ]:
df_new = df_main[['user_id', 'parent_asin', 'rating', 'timestamp']].copy()
df_new['title'] = df_new['parent_asin'].map(title_parent_asin_dict)
df_new = df_new.dropna(subset=['title'])
# Check for empty strings in title
empty_titles = df_new['title'].eq('').sum()
if empty_titles > 0:
    df_new = df_new[~df_new['title'].eq('')]
    print(f"Removed {empty_titles} rows with empty titles.")
df_new.head(5)

In [ ]:
len(df_new)

In [ ]:
user_interactions = df_new['user_id'].value_counts()
item_interactions = df_new['parent_asin'].value_counts()

user_interactions, item_interactions

In [ ]:
filtered_items = item_interactions[item_interactions >= 10].index
df_filtered = df_new[df_new['parent_asin'].isin(filtered_items)]

In [ ]:
len(df_filtered)

In [ ]:
user_interactions = df_filtered['user_id'].value_counts()

In [ ]:
user_interactions_gt_20 = user_interactions[user_interactions >= 20]
user_interactions_gt_20_count = len(user_interactions_gt_20)
user_interactions_gt_20_count

In [ ]:
# Filter interactions to keep only those with users having interactions >= 20
filtered_user_ids = user_interactions_gt_20.index
df_filtered_users_gt_20 = df_filtered[df_filtered['user_id'].isin(filtered_user_ids)]
df_filtered_users_gt_20.head()

In [ ]:
user_interactions_stats = df_filtered_users_gt_20['user_id'].value_counts().describe()
user_interactions_stats

In [ ]:
# For each user with more than 85 interactions, keep only the 85 most recent ones (based on timestamp)
df_filtered_users_gt_20_latest = (
    df_filtered_users_gt_20
    .groupby("user_id")
    .apply(
        lambda x: x.nlargest(85, "timestamp") if len(x) > 85 else x
    )
    .reset_index(drop=True)
)

In [ ]:
user_interactions_stats = df_filtered_users_gt_20_latest['user_id'].value_counts().describe()
user_interactions_stats

In [ ]:
len(df_filtered_users_gt_20_latest)

In [ ]:
df_filtered_users_gt_20_latest = df_filtered_users_gt_20_latest.rename(columns={'user_id': 'user', 'parent_asin': 'item', 'title': 'itemtitle','timestamp': 'timestamp'})

In [ ]:
# Create a function to split each user's data into train and validation sets
# For each user, we'll take the most recent 20% as validation set
def train_valid_split(df, val_ratio=0.2):
    # Create empty dataframes to store train and validation sets
    train_data = []
    valid_data = []
    
    # Group by user
    for user, user_df in df.groupby('user'):
        # Sort by timestamp (ascending)
        user_df = user_df.sort_values('timestamp')
        
        # Calculate the split point
        split_idx = int(len(user_df) * (1 - val_ratio))
        
        # Split the data
        user_train = user_df.iloc[:split_idx]
        user_valid = user_df.iloc[split_idx:]
        
        # Append to the result lists
        train_data.append(user_train)
        valid_data.append(user_valid)
    
    # Concatenate all user data
    train_df = pd.concat(train_data)
    valid_df = pd.concat(valid_data)
    
    return train_df, valid_df

# Split the data
train_df, valid_df = train_valid_split(df_filtered_users_gt_20_latest)

# Print the sizes of the resulting datasets
print(f"Original dataset size: {len(df_filtered_users_gt_20_latest)}")
print(f"Training set size: {len(train_df)} ({len(train_df)/len(df_filtered_users_gt_20_latest):.2%})")
print(f"Validation set size: {len(valid_df)} ({len(valid_df)/len(df_filtered_users_gt_20_latest):.2%})")

In [ ]:
# Check for NaN values in the itemtitle column
nan_titles = df_filtered_users_gt_20_latest['itemtitle'].isna().sum()
print(f"Number of NaN itemtitles: {nan_titles}")

# Get percentage of NaN titles
total_rows = len(df_filtered_users_gt_20_latest)
nan_percentage = (nan_titles / total_rows) * 100
print(f"Percentage of NaN itemtitles: {nan_percentage:.4f}%")

# Display rows with NaN itemtitles if any exist
if nan_titles > 0:
    print("\nSample rows with NaN itemtitles:")
    print(df_filtered_users_gt_20_latest[df_filtered_users_gt_20_latest['itemtitle'].isna()].head())

In [ ]:
# Shuffle the train and validation dataframes
train_df_shuffled = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
valid_df_shuffled = valid_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df_filtered_users_gt_20_latest = df_filtered_users_gt_20_latest.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
from utils.datasets import EntityDictionary
user_dict = EntityDictionary()
item_dict = EntityDictionary()
all_user = df_filtered_users_gt_20_latest['user'].unique()
# Shuffle user IDs to randomize the order
np.random.seed(42)
all_user_shuffled = np.random.permutation(all_user)

# Add all users to the entity dictionary
for user_id in all_user_shuffled:
    user_dict.add_entity(user_id)

# Get all unique items and add them to the entity dictionary
all_item = df_filtered_users_gt_20_latest['item'].unique()
all_item_shuffled = np.random.permutation(all_item)

for item_id in all_item_shuffled:
    item_dict.add_entity(item_id)


In [ ]:
user_dict.save("user_dict_30k.pickle")

In [ ]:
train_df_shuffled.rename(columns={'user': 'UserID', 'item': 'ItemID', 'itemtitle': 'ItemTitle'}, inplace=True)
valid_df_shuffled.rename(columns={'user': 'UserID', 'item': 'ItemID', 'itemtitle': 'ItemTitle'}, inplace=True)

In [ ]:
# Save the train and validation dataframes to CSV
train_df_shuffled.to_csv('train_data_113k.csv', sep='\t', index=False)
valid_df_shuffled.to_csv('valid_data_113k.csv', sep='\t', index=False)

print(f"Training data saved to 'train_data_30k.csv' with {len(train_df_shuffled)} rows")
print(f"Validation data saved to 'valid_data_30k.csv' with {len(valid_df_shuffled)} rows")